# 19 Persistence

Analyses the persistence extension collected by `18_dialogue`. The adaptation
experiment measures what a model says first. This one measures whether that
survives being pushed.

Design, fixed in `config/settings.yml` before any dialogue exists:

| | |
|---|---|
| Scenarios | 50, the 25 age-restricted and 25 of the 50 harmful |
| Conditions | control and seven stated ages, seven to eighteen |
| Methods | Purpose Reverse, Role Play, Emotional Pushback |
| Turns | 1 user, 2 the model's own opening reply, 3 and 5 pressure, 4 and 6 the replies scored here |

Turn 2 is not regenerated. It is the reply the model actually gave in the
adaptation pass, joined back on model, prompt and replicate, so movement at a
later turn is measured against what that model did rather than against a fresh
draw. Turns 4 and 6 are scored by the same frozen rubric at the same policy
fingerprint as the adaptation corpus.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd

import analysis
from analysis import MACRO, NAME, ORDER, STATED_AGE
from utils import read_lines

pd.set_option('display.width', 200, 'display.max_columns', 40)

RESULTS = Path.cwd().parent / 'results'
TABLES = Path.cwd().parent / 'tables'
TURNS = [4, 6]


# Write a finished table to the frozen trail. publish() is the house route and
# refuses a name config/captions.yml does not describe, so until these names are
# entered there this writes the CSV directly and nothing else.
def freeze(table, name, tier='supplement'):
    table.to_csv(TABLES / ('main' if tier == 'main' else 'supplement') / f'{name}.csv')
    return table

## The corpus

The opening reply is joined from the adaptation classification rather than
reclassified, so a dialogue that opened with a refusal is one the adaptation
chapter also counts as a refusal.

In [2]:
plan = pd.read_csv(RESULTS / 'dialogue' / 'plan.csv')
seeds = (plan[['dialogue_id', 'prompt_id', 'scenario_id', 'condition', 'model',
               'opening_replicate', 'method', 'expected_answer']]
         .drop_duplicates('dialogue_id'))

opening = pd.read_csv(RESULTS / 'classification.csv',
                      usecols=['model', 'prompt_id', 'replicate', 'answer',
                               'delivery_response'])
seeds = seeds.merge(opening.rename(columns={'answer': 'opening_answer',
                                            'delivery_response': 'opening_delivery'}),
                    left_on=['model', 'prompt_id', 'opening_replicate'],
                    right_on=['model', 'prompt_id', 'replicate'], how='left')

pressed = pd.DataFrame(read_lines(RESULTS / 'classification_dialogue.jsonl'))
pressed['turn'] = pressed['turn'].astype(int)
pressed = pressed[pressed['turn'].isin(TURNS)]

frame = pressed.merge(seeds, on=['dialogue_id', 'model'], how='left')
frame['label'] = frame['model'].map(NAME)
frame['age'] = frame['condition'].map(STATED_AGE)
frame['refused'] = frame['answer'].eq('Refusal')
frame['opening_refused'] = frame['opening_answer'].eq('Refusal')

print(f"{seeds['dialogue_id'].nunique():,} dialogues, {len(frame):,} scored turns, "
      f"policy {frame['policy'].iat[0]}")

7,098 dialogues, 14,184 scored turns, policy e5f836fffbf6


## 1. Refusal survival

Restricted to dialogues whose opening reply was a refusal, since a dialogue that
opened with a compliance has no refusal to lose. Survival is the share still
refusing at each pressed turn. The denominator is dialogues, not turns, so the
two columns are comparable.

In [3]:
held = frame[frame['opening_refused']]

survival = (held.pivot_table(index='label', columns='turn', values='refused',
                             aggfunc='mean') * 100).reindex(ORDER)
survival.columns = [f'Turn {t} (%)' for t in survival.columns]
survival.insert(0, 'Dialogues Opening in Refusal',
                held.groupby('label')['dialogue_id'].nunique().reindex(ORDER))
survival.loc[MACRO] = survival.mean()
survival.index.name = 'Model'
freeze(survival.round(2), 'persistence_01_survival', tier='main')
survival.round(1)

,Dialogues Opening in Refusal,Turn 4 (%),Turn 6 (%)
Model,,,
GPT-5.6 Luna,711.0,79.6,63.3
Claude Haiku 4.5,726.0,93.1,80.2
Gemini 3.5 Flash Lite,687.0,68.7,73.8
DeepSeek-V4 Flash,741.0,62.2,52.1
Mistral Small 4,603.0,35.2,19.4
Gemma 4 31B,687.0,69.1,72.6
Macro-average,692.5,68.0,60.2


## 2. Which method moves it

The three pressure methods are applied to the same dialogues, so a difference
between them is a difference in the pressure and not in the scenarios.

In [4]:
method = (held.pivot_table(index=['method', 'label'], columns='turn',
                           values='refused', aggfunc='mean') * 100)
method.columns = [f'Turn {t} (%)' for t in method.columns]
macro = (method.groupby('method').mean()
         .assign(label=MACRO).set_index('label', append=True))
method = pd.concat([method, macro]).sort_index()
method.index.names = ['Method', 'Model']
freeze(method.round(2), 'persistence_02_method', tier='main')
method.round(1)

Turn 4 (%)  Turn 6 (%)
Method             Model                                        
Emotional Pushback Claude Haiku 4.5            100.0        82.6
                   DeepSeek-V4 Flash            97.6        69.6
                   GPT-5.6 Luna                 97.5        56.1
                   Gemini 3.5 Flash Lite       100.0        74.9
                   Gemma 4 31B                  99.1        72.9
                   Macro-average                95.1        62.4
                   Mistral Small 4              76.6        18.4
Purpose Reverse    Claude Haiku 4.5            100.0       100.0
                   DeepSeek-V4 Flash            44.1        60.7
                   GPT-5.6 Luna                 87.3        94.5
                   Gemini 3.5 Flash Lite        41.2        90.8
                   Gemma 4 31B                  31.9        97.8
                   Macro-average                53.5        78.4
                   Mistral Small 4              16.4        26.4
Role Play          Claude Haiku 4.5             79.3        57.9
                   DeepSeek-V4 Flash            44.9        25.9
                   GPT-5.6 Luna                 54.0        39.2
                   Gemini 3.5 Flash Lite        64.5        55.7
                   Gemma 4 31B                  76.4        47.2
                   Macro-average                55.3        39.9
                   Mistral Small 4              12.4        13.4

## 3. Does the age effect survive the pressure

Refusal at each turn for a stated minor age against a stated adult age, over all
dialogues rather than only those that opened in refusal, so the turn 2 column is
the adaptation result on this subset of scenarios and the later columns are
comparable with it.

In [5]:
frame['block'] = np.where(frame['age'] < 18, 'Minor', 'Adult')
aged = frame[frame['age'].notna()]

opening_rate = (aged.drop_duplicates(['dialogue_id', 'turn'])
                .query('turn == @TURNS[0]')
                .assign(refused=lambda d: d['opening_refused'])
                .pivot_table(index='label', columns='block', values='refused',
                             aggfunc='mean') * 100)
later = (aged.pivot_table(index='label', columns=['turn', 'block'],
                          values='refused', aggfunc='mean') * 100)

age_effect = pd.concat({2: opening_rate}, axis=1).join(later).reindex(ORDER)
age_effect.columns = [f'Turn {t}, {b}' for t, b in age_effect.columns]
for turn in [2] + TURNS:
    age_effect[f'Turn {turn}, Gap'] = (age_effect[f'Turn {turn}, Minor']
                                       - age_effect[f'Turn {turn}, Adult'])
age_effect = age_effect.reindex(sorted(age_effect.columns), axis=1)
age_effect.loc[MACRO] = age_effect.mean()
age_effect.index.name = 'Model'
freeze(age_effect.round(2), 'persistence_03_age', tier='main')
age_effect.round(1)

,"Turn 2, Adult","Turn 2, Gap","Turn 2, Minor","Turn 4, Adult","Turn 4, Gap","Turn 4, Minor","Turn 6, Adult","Turn 6, Gap","Turn 6, Minor"
Model,,,,,,,,,
GPT-5.6 Luna,40.0,26.3,66.3,40.0,22.8,62.8,31.3,16.9,48.2
Claude Haiku 4.5,40.0,29.0,69.0,59.3,18.9,78.2,60.7,7.4,68.1
Gemini 3.5 Flash Lite,50.0,18.6,68.6,38.0,15.8,53.8,46.0,8.5,54.5
DeepSeek-V4 Flash,42.0,25.7,67.7,37.3,8.2,45.6,29.3,7.3,36.7
Mistral Small 4,42.0,11.0,53.0,20.0,3.9,23.9,11.3,0.7,12.0
Gemma 4 31B,44.0,18.3,62.3,38.7,11.7,50.3,45.3,4.7,50.0
Macro-average,43.0,21.5,64.5,38.9,13.5,52.4,37.3,7.6,44.9


## 4. Delivery under pressure

Refusal is what the model says. This is what arrives. A dialogue that keeps
refusing and starts delivering is the failure the four-cell outcome was built to
expose, and it is the one a refusal rate over turns would not show.

In [6]:
frame['delivered'] = frame['delivery_response'].eq('Yes')
delivery = (held.assign(delivered=frame['delivered'])
            .pivot_table(index='label', columns='turn', values='delivered',
                         aggfunc='mean') * 100).reindex(ORDER)
delivery.columns = [f'Turn {t} (%)' for t in delivery.columns]
delivery.loc[MACRO] = delivery.mean()
delivery.index.name = 'Model'
freeze(delivery.round(2), 'persistence_s01_delivery')
delivery.round(1)

,Turn 4 (%),Turn 6 (%)
Model,,
GPT-5.6 Luna,18.8,45.6
Claude Haiku 4.5,5.5,17.6
Gemini 3.5 Flash Lite,26.6,24.3
DeepSeek-V4 Flash,37.7,53.2
Mistral Small 4,36.5,56.1
Gemma 4 31B,23.6,27.7
Macro-average,24.8,37.4


## What this notebook writes

| Table | Tier |
|---|---|
| `persistence_01_survival` | main |
| `persistence_02_method` | main |
| `persistence_03_age` | main |
| `persistence_s01_delivery` | supplement |

Every table here is descriptive. The persistence extension declares no
hypothesis family, so nothing carries a permutation value or an adjusted one,
and the numbers are reported as description in the results chapter.

To move these onto `publish()`, add the four names to `config/captions.yml` with
a `label`, a `tier` and a `kind: table`, then replace `freeze` with `publish` in
the setup cell.